# 03 – Model & priorities
Input: `data_clean/reviews_laundry_features.csv`

Logistic regression of low rating on attribute flags, odds ratios, importance-performance matrix.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

d = pd.read_csv("data_clean/reviews_laundry_features.csv")
A = [c for c in d.columns if c.startswith("a_")]
m = d.dropna(subset=["low"]).copy()
m["verified"] = m.verified_purchase.astype(int)
len(m)

## 1. Logistic regression (all brands)

In [ ]:
f = "low ~ " + " + ".join(A) + " + C(brand, Treatment('LG')) + verified + C(year)"
logit = smf.logit(f, data=m).fit(disp=0)
ci = logit.conf_int()
orr = pd.DataFrame({"OR": np.exp(logit.params), "lo": np.exp(ci[0]), "hi": np.exp(ci[1]), "p": logit.pvalues}).round(3)
orr.loc[A].sort_values("OR", ascending=False)

In [ ]:
# Forest plot of odds ratios
t = orr.loc[A].sort_values("OR")
plt.figure(figsize=(7, 4))
plt.errorbar(t.OR, range(len(t)), xerr=[t.OR - t.lo, t.hi - t.OR], fmt="o", capsize=3)
plt.axvline(1, color="grey", ls="--"); plt.yticks(range(len(t)), [i[2:] for i in t.index]); plt.xscale("log")
plt.xlabel("Odds ratio for a low rating (log scale)"); plt.tight_layout(); plt.savefig("figures/03_odds_ratios.png", dpi=150)

## 2. LG only and LG vs. competitors

In [ ]:
lg = m[m.is_lg == 1]
f_lg = "low ~ " + " + ".join(A) + " + verified + C(year)"
lg_fit = smf.logit(f_lg, data=lg).fit(disp=0)
pd.DataFrame({"OR_LG": np.exp(lg_fit.params[A]), "p": lg_fit.pvalues[A]}).round(3)

In [ ]:
# Mention rate of each attribute: LG vs. competitors, among low reviews
lowr = m[m.low == 1]
cmp_ = lowr.groupby("is_lg")[A].mean().T.rename(columns={0: "competitors", 1: "LG"}).round(3)
cmp_["gap"] = cmp_.LG - cmp_.competitors
cmp_

## 3. Importance-performance matrix
x = share of LG low reviews mentioning the attribute (frequency). y = log odds ratio (harm). Top-right = priority.

In [ ]:
pri = pd.DataFrame({"freq": lowr[lowr.is_lg == 1][A].mean(), "logOR": logit.params[A]})
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(pri.freq, pri.logOR, s=80)
for k, r in pri.iterrows():
    ax.annotate(k[2:], (r.freq, r.logOR), xytext=(5, 5), textcoords="offset points")
ax.axhline(0, color="grey", ls="--"); ax.axvline(pri.freq.median(), color="grey", ls=":")
ax.set_xlabel("Frequency in LG low-rated reviews"); ax.set_ylabel("log odds ratio (harm)")
plt.tight_layout(); plt.savefig("figures/03_importance_performance.png", dpi=150)
pri.sort_values("logOR", ascending=False)

## 4. Robustness (fill in)
- [ ] `low_strict` (1-star only)
- [ ] 2016+ only
- [ ] refrigerator_freezer (rerun 01-03 with `CATEGORY` changed)
- [ ] OLS on `review_rating`